## Experiments Growing Arrays

#### Conclusion
*for growing:* We want to pre-allocate a bucket big enough to hold the largest expected number of items and hope we make it big enough. Then, insert new items in place.

*for deleting:* We want to have a boolean that matches the maximum size of the bucket. Values need to be initialized to *false*. Then, we set each item to true for deleting from the bucket. Then we will have to expand the bucket again and expand the filt of the bucket back to maximum length. But, expanding by resize! is really slow.  So, deleting really doesn't work.  Instead, we want to zero out the elements we don't want any more. But, then we need to sort to put the zeros at the end (or the beginning--but the end is easier).

*for zeroing and sorting:* Both operations are pretty slow.

*conclusion:* We have to conclude that the bucket strategy doesn't work if the bucket is big and/or the number of items we have in the bucket changes each time. The fastest thing is to change values of the vector in place and then use findall to generate indexes to each value.

In [80]:
using BenchmarkTools
using StatsBase
using Random

In [16]:
# naive push!

function grow_by_push(arr, item, n_add=100)
#     arr = copy(arr)
    for i in 1:n_add
        push!(arr, item)
    end
    
    return arr
end
        
    

grow_by_push (generic function with 3 methods)

In [42]:
# append and return, append!

function append_and_return(arr1, arr2)
    arr_in = copy(arr1)
    append!(arr_in, arr2)
    return arr_in
end

append_and_return (generic function with 1 method)

In [43]:
function append(arr1, arr2)
    append!(arr1, arr2)
    return
end

append (generic function with 1 method)

In [54]:
function insertat_loop!(arr1, arr2, position)
    insat = position
    for i in 1:length(arr2)
        arr1[insat] = arr2[i]
        insat += 1
    end
end

insertat_loop! (generic function with 1 method)

In [47]:
function insertat!(arr1, arr2, position)
    pos_end = position + length(arr2) -1
    arr1[position:pos_end] = arr2
end

insertat! (generic function with 1 method)

In [40]:
arr, item = setup(10_000, Float64)
arr = copy(arr)
@benchmark grow_by_push(copy(arr), item, 5_000) setup=(arr)

BenchmarkTools.Trial: 
  memory estimate:  156.33 KiB
  allocs estimate:  3
  --------------
  minimum time:     39.465 μs (0.00% GC)
  median time:      84.055 μs (0.00% GC)
  mean time:        86.801 μs (2.77% GC)
  maximum time:     2.510 ms (98.21% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [35]:

@benchmark append_and_return(arr1, arr2) setup=(arr1=zeros(Float64, 10_000); arr2=ones(Float64, 5_000))

BenchmarkTools.Trial: 
  memory estimate:  156.33 KiB
  allocs estimate:  3
  --------------
  minimum time:     10.528 μs (0.00% GC)
  median time:      85.311 μs (0.00% GC)
  mean time:        105.137 μs (16.56% GC)
  maximum time:     174.139 ms (99.97% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [41]:

@benchmark append(arr1, arr2) setup=(arr1=zeros(Float64, 10_000); arr2=ones(Float64, 5_000))

BenchmarkTools.Trial: 
  memory estimate:  58.59 KiB
  allocs estimate:  0
  --------------
  minimum time:     3.970 μs (0.00% GC)
  median time:      24.580 μs (0.00% GC)
  mean time:        25.393 μs (6.12% GC)
  maximum time:     682.361 μs (98.91% GC)
  --------------
  samples:          10000
  evals/sample:     4

In [39]:

@benchmark append!(arr1, arr2) setup=(arr1=zeros(Float64, 10_000); arr2=ones(Float64, 5_000))

BenchmarkTools.Trial: 
  memory estimate:  58.59 KiB
  allocs estimate:  0
  --------------
  minimum time:     3.857 μs (0.00% GC)
  median time:      24.220 μs (0.00% GC)
  mean time:        24.974 μs (6.01% GC)
  maximum time:     613.806 μs (99.16% GC)
  --------------
  samples:          10000
  evals/sample:     4

In [51]:
@benchmark zeros(Float64, 15_000)

BenchmarkTools.Trial: 
  memory estimate:  117.27 KiB
  allocs estimate:  2
  --------------
  minimum time:     6.934 μs (0.00% GC)
  median time:      43.798 μs (0.00% GC)
  mean time:        53.238 μs (18.73% GC)
  maximum time:     1.131 ms (96.05% GC)
  --------------
  samples:          10000
  evals/sample:     4

In [52]:
@benchmark Vector{Float64}(undef, 15_000) # a lot faster to define the memory w/o filling it

BenchmarkTools.Trial: 
  memory estimate:  117.27 KiB
  allocs estimate:  2
  --------------
  minimum time:     1.896 μs (54.35% GC)
  median time:      2.062 μs (53.54% GC)
  mean time:        2.466 μs (62.08% GC)
  maximum time:     5.429 μs (74.12% GC)
  --------------
  samples:          3809
  evals/sample:     530

In [50]:
@benchmark insertat!(arr1, arr2, 10_001) setup=(arr1 = zeros(Float64,15_000); arr2=ones(Float64, 5_000))


BenchmarkTools.Trial: 
  memory estimate:  0 bytes
  allocs estimate:  0
  --------------
  minimum time:     794.010 ns (0.00% GC)
  median time:      869.485 ns (0.00% GC)
  mean time:        880.339 ns (0.00% GC)
  maximum time:     2.119 μs (0.00% GC)
  --------------
  samples:          10000
  evals/sample:     103

In [55]:
@benchmark insertat_loop!(arr1, arr2, 10_001) setup=(arr1 = zeros(Float64,15_000); arr2=ones(Float64, 5_000))

BenchmarkTools.Trial: 
  memory estimate:  0 bytes
  allocs estimate:  0
  --------------
  minimum time:     3.317 μs (0.00% GC)
  median time:      3.413 μs (0.00% GC)
  mean time:        3.501 μs (0.00% GC)
  maximum time:     21.773 μs (0.00% GC)
  --------------
  samples:          10000
  evals/sample:     8

## Deleting strategies

In [67]:
d = @benchmarkable deleteat!(a, delbool) setup=(a = collect(1:15_000); 
                                 delbool = rand([false, false, false, false, false, true],15_000))

run(d)

BenchmarkTools.Trial: 
  memory estimate:  0 bytes
  allocs estimate:  0
  --------------
  minimum time:     8.092 μs (0.00% GC)
  median time:      8.551 μs (0.00% GC)
  mean time:        8.758 μs (0.00% GC)
  maximum time:     57.274 μs (0.00% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [69]:
# delete by index without sorting (set indices in sorted order...   
# the application will need to sort, but this is for one benchmark

e = @benchmarkable deleteat!(a, delidx) setup=(a = collect(1:15_000); 
                                 delidx = collect(2500:2:7500))
run(e)

BenchmarkTools.Trial: 
  memory estimate:  0 bytes
  allocs estimate:  0
  --------------
  minimum time:     6.033 μs (0.00% GC)
  median time:      6.484 μs (0.00% GC)
  mean time:        6.594 μs (0.00% GC)
  maximum time:     55.996 μs (0.00% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [97]:
# delete by index with sorting--try different ways to sort
f = @benchmarkable deleteat!(a, sort!(delidx)) setup=(a = collect(1:15_000); 
                                 delidx = shuffle(collect(2500:2:7500)))
run(f)

BenchmarkTools.Trial: 
  memory estimate:  0 bytes
  allocs estimate:  0
  --------------
  minimum time:     97.350 μs (0.00% GC)
  median time:      105.599 μs (0.00% GC)
  mean time:        107.327 μs (0.00% GC)
  maximum time:     283.753 μs (0.00% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [99]:
# evaluate different ways to sort: sort and assign
g = @benchmarkable x = sort(delidx) setup=(delidx = shuffle(collect(2500:2:7500)))
run(g)

BenchmarkTools.Trial: 
  memory estimate:  19.64 KiB
  allocs estimate:  2
  --------------
  minimum time:     92.907 μs (0.00% GC)
  median time:      101.424 μs (0.00% GC)
  mean time:        105.312 μs (1.53% GC)
  maximum time:     4.728 ms (97.72% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [100]:
# evaluate different ways to sort: sort in place--> this is better than allocating a new vector
g = @benchmarkable sort!(delidx) setup=(delidx = shuffle(collect(2500:2:7500)))
run(g)

BenchmarkTools.Trial: 
  memory estimate:  0 bytes
  allocs estimate:  0
  --------------
  minimum time:     89.082 μs (0.00% GC)
  median time:      100.811 μs (0.00% GC)
  mean time:        102.175 μs (0.00% GC)
  maximum time:     334.425 μs (0.00% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [103]:

h = @benchmarkable delidx[sortperm(delidx)] setup=(delidx = shuffle(collect(2500:2:7500)))

run(h)

BenchmarkTools.Trial: 
  memory estimate:  39.30 KiB
  allocs estimate:  5
  --------------
  minimum time:     117.272 μs (0.00% GC)
  median time:      126.412 μs (0.00% GC)
  mean time:        133.649 μs (2.63% GC)
  maximum time:     5.451 ms (97.33% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [118]:
ix = Vector{Int}(undef, 2501)
i = @benchmarkable delidx[sortperm!($ix,delidx)] setup=(delidx = shuffle(collect(2500:2:7500)))

run(i)

BenchmarkTools.Trial: 
  memory estimate:  19.66 KiB
  allocs estimate:  3
  --------------
  minimum time:     115.530 μs (0.00% GC)
  median time:      126.219 μs (0.00% GC)
  mean time:        130.016 μs (1.28% GC)
  maximum time:     5.303 ms (97.48% GC)
  --------------
  samples:          10000
  evals/sample:     1

### Resizing to grow a bucket

In [120]:
j = @benchmarkable resize!(bucket, 15_000) setup = (bucket = rand([1,2,3],12500))
run(j)

BenchmarkTools.Trial: 
  memory estimate:  97.66 KiB
  allocs estimate:  1
  --------------
  minimum time:     4.857 μs (0.00% GC)
  median time:      43.350 μs (0.00% GC)
  mean time:        42.875 μs (0.00% GC)
  maximum time:     227.326 μs (0.00% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [121]:
k = @benchmarkable append!(bucket, zeros(Int,2500)) setup = (bucket = rand([1,2,3],12500))
run(k)

BenchmarkTools.Trial: 
  memory estimate:  117.30 KiB
  allocs estimate:  3
  --------------
  minimum time:     7.358 μs (0.00% GC)
  median time:      53.397 μs (0.00% GC)
  mean time:        52.543 μs (0.52% GC)
  maximum time:     2.740 ms (99.25% GC)
  --------------
  samples:          10000
  evals/sample:     1

In [122]:
l = @benchmarkable append!(bucket, Vector{Int}(undef,2500)) setup = (bucket = rand([1,2,3],12500))
run(l)

BenchmarkTools.Trial: 
  memory estimate:  117.30 KiB
  allocs estimate:  3
  --------------
  minimum time:     7.003 μs (0.00% GC)
  median time:      54.033 μs (0.00% GC)
  mean time:        53.537 μs (0.75% GC)
  maximum time:     4.053 ms (99.61% GC)
  --------------
  samples:          10000
  evals/sample:     1

### zeroing and sorting the bucket

In [126]:
a = collect(1:15_000)
zz = collect(2500:2:7500)
a[zz] .= 0
sort!(a, rev=true)

15000-element Vector{Int64}:
 15000
 14999
 14998
 14997
 14996
 14995
 14994
 14993
 14992
 14991
 14990
 14989
 14988
     ⋮
     0
     0
     0
     0
     0
     0
     0
     0
     0
     0
     0
     0

In [135]:
m = @benchmarkable sort!(a, rev=true)  setup=(a = collect(1:15_000); zz = collect(2500:2:7500); a[zz].= 0)
run(m)

BenchmarkTools.Trial: 
  memory estimate:  0 bytes
  allocs estimate:  0
  --------------
  minimum time:     1.270 ms (0.00% GC)
  median time:      1.311 ms (0.00% GC)
  mean time:        1.333 ms (0.00% GC)
  maximum time:     2.237 ms (0.00% GC)
  --------------
  samples:          3536
  evals/sample:     1

### Instead of a bucket, change the values in place and then filter

In [145]:
n = @benchmarkable (findall(a .== 2)) setup=(a = fill(1, 15_000);idx = collect(2500:2:7500);a[idx] .= 2; )


run(n)

BenchmarkTools.Trial: 
  memory estimate:  25.84 KiB
  allocs estimate:  5
  --------------
  minimum time:     8.552 μs (0.00% GC)
  median time:      9.626 μs (0.00% GC)
  mean time:        9.948 μs (0.00% GC)
  maximum time:     47.577 μs (0.00% GC)
  --------------
  samples:          10000
  evals/sample:     1